In [0]:
from pyspark.sql.functions import *

### Checking Connection Test

In [0]:
dbutils.fs.ls("/Volumes/hm_adb_caresync_workspace/landing/landing-volume/neon_postgres/hospitals/2026-10-07_02:00:14/")

In [0]:
# Reading Parquet file from ADLS -> Landing

df = spark.read.format('parquet')\
    .load('/Volumes/hm_adb_caresync_workspace/landing/landing-volume/neon_postgres/hospitals/2026-10-07_02:00:14/')

df.display()


## CREATEING PARAMETERS

In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")


source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")


In [0]:
landing_path = f'/Volumes/hm_adb_caresync_workspace/landing/landing-volume/{source_system}/{table_name}/{landing_timestamp}/'

try:
    dbutils.fs.ls(landing_path)
    print(f"Files found at: {landing_path}")
except Exception:
    print(f"No new files received at path: {landing_path}")
    dbutils.notebook.exit("No new files received. Exiting notebook without failure.")

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)
     
landing_df.display()

In [0]:
from pyspark.sql.functions import *

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
    .withColumn("insert_timestamp", current_timestamp())

updated_df.display()

In [0]:
updated_df.write.mode('append')\
    .saveAsTable(f'hm_adb_caresync_workspace.bronze.{table_name}')